In [0]:
import pandas as pd                                                   # Import pandas for data cleaning
import numpy as np                                                    # Import Numpy for Maths functions
import matplotlib.pyplot as plt                                       # Import Matplot for Viz functions
import seaborn as sns                                                 # Import Seaborn for visualization
import matplotlib.pyplot as plt                                       # Import matplotlib library for visualization
import plotly.express as px                                           # Import plotly library for visualization
import plotly.graph_objects as go

from sklearn.ensemble import IsolationForest                          # EDA-Isolation Forest Analysis Functions

from pyspark.sql import functions as F
from pyspark.sql.functions import col, StringType, NumericType                  # TableFunctions
from pyspark.sql.functions import mean, min, max, stddev, count, sum as _sum    # MathsFunctions
from pyspark.sql.functions import to_date, year, month, datediff                # DateFunctions
from pyspark.sql.functions import abs                                           # OtherFunctions

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

from sklearn.linear_model import LinearRegression                              # LinearRegression Analysis Functions
from sklearn.metrics import r2_score, mean_squared_error                       # LinearRegression Analysis Functions

from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder    # Classification Analysis Functions
from pyspark.ml import Pipeline                                                 # Classification Analysis Functions
from sklearn.linear_model import LogisticRegression                             # Classification Analysis Functions
from sklearn.metrics import mean_squared_error, r2_score                        # Classification Analysis Functions
from sklearn.impute import SimpleImputer                                        # Classification Analysis Functions
from sklearn.preprocessing import LabelEncoder                                  # Classification Analysis Functions
from sklearn.preprocessing import OneHotEncoder                                 # Classification Analysis Functions
from sklearn.preprocessing import StandardScaler                                # Classification Analysis Functions
from sklearn.model_selection import train_test_split                            # Classification Analysis Functions
from sklearn.metrics import accuracy_score, confusion_matrix                    # Classification Analysis Functions
from sklearn.metrics import classification_report                               # Classification Analysis Functions
from sklearn.metrics import roc_curve, auc                                      # Classification Analysis Functions

from sklearn.tree import DecisionTreeClassifier                                 # DecisionTree Analysis Functions
from sklearn.tree import DecisionTreeRegressor                                  # DecisionTree Analysis Functions

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier                             # RandomForest Analysis Functions

from sklearn.cluster import KMeans                                              # KMeans Cluster Analysis Functions


In [0]:
df = spark.read.table("indianhousing.silver.indian_housing").toPandas()

# DEFINE
X = df[["house_size"]]
y = df["price"]

In [0]:
#SPLIT DATA
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

#MODEL
model = LinearRegression()
model.fit(X_train,y_train)

#PREDICTIONS
y_pred = model.predict(X_test)

#EVOLUATION
print("INTERCEPT:",model.intercept_)
print("COEFFICIENT:",model.coef_)
print("R2:",r2_score(y_test,y_pred))
print("RMSE:",np.sqrt(mean_squared_error(y_test,y_pred)))

#------------------------------------------------------------------------------------------------
full_model = LinearRegression()
full_model.fit(X,y)
linear_pred = full_model.predict(X)

#EVOLUATION
print("INTERCEPT:",full_model.intercept_)
print("COEFFICIENT:",full_model.coef_)
print("R2:",r2_score(y,linear_pred))
print("RMSE (full):",np.sqrt(mean_squared_error(y,linear_pred)))

df["linear_predictions"] = linear_pred

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

# DEFINE
X = df[["house_size"]]
y = df["price"]

# Polynomial transform (degree=2)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# ---------------------- DEV (Train/Test Evaluation) ----------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Transform train & test
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# MODEL
dev_model = LinearRegression()
dev_model.fit(X_train_poly, y_train)

# PREDICTIONS
y_pred_dev = dev_model.predict(X_test_poly)

# EVALUATION
print("DEV Intercept:", dev_model.intercept_)
print("DEV Coefficients:", dev_model.coef_)
print("DEV R2:", r2_score(y_test, y_pred_dev))
print("DEV RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dev)))

# ---------------------- PROD (Full Dataset Fit) ----------------------
prod_model = LinearRegression()
prod_model.fit(X_poly, y)

# PREDICTIONS on full dataset
y_pred_full = prod_model.predict(X_poly)

# EVALUATION
print("PROD Intercept:", prod_model.intercept_)
print("PROD Coefficients:", prod_model.coef_)
print("PROD R2:", r2_score(y, y_pred_full))
print("PROD RMSE:", np.sqrt(mean_squared_error(y, y_pred_full)))

# Save predictions to DataFrame
df["poly_predictions"] = y_pred_full


In [0]:
# --------------------------- DEV(Train/test Evoluation) ---------------------------
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = DecisionTreeRegressor(random_state=42)

model.fit(X_train, y_train)

model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)


# PREDICTIONS
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# EVALUATION
print("R2_train:", r2_score(y_train, y_pred_train))
print("R2_test:", r2_score(y_test, y_pred_test))
print("RMSE_train:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("RMSE_test:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

# --------------------------- FULL DATASET -----------------------------------------
full_model = DecisionTreeRegressor(random_state=42)
full_model.fit(X,y)

dt_predictions = full_model.predict(X)

print("R2_full:",r2_score(y,dt_predictions))
print("RMSE(full):",np.sqrt(mean_squared_error(y,dt_predictions)))

df["dt_predictions"] = dt_predictions

In [0]:
# --------------------------- DEV(Train/test Evoluation) ---------------------------
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

# PREDICTIONS
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# EVALUATION
print("R2_train:", r2_score(y_train, y_pred_train))
print("R2_test:", r2_score(y_test, y_pred_test))
print("RMSE_train:", np.sqrt(mean_squared_error(y_train, y_pred_train)))
print("RMSE_test:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

# --------------------------- FULL DATASET -----------------------------------------
full_model = RandomForestRegressor(random_state=42)
full_model.fit(X,y)

rf_predictions = full_model.predict(X)

print("R2_full:",r2_score(y,rf_predictions))
print("RMSE(full):",np.sqrt(mean_squared_error(y,rf_predictions)))


df["rf_predictions"]=rf_predictions

In [0]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# FEATURES (example: house_size, price)
X = df[["house_size", "price"]]

# Scale data (important for KMeans)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------------------- KMeans Model ----------------------
# Choose number of clusters (k=3 as example)
kmeans = KMeans(n_clusters=5, random_state=42)
df["K_Means_Cluster"] = kmeans.fit_predict(X_scaled)

# Cluster centers (scaled → inverse transform for original scale)
centers = scaler.inverse_transform(kmeans.cluster_centers_)


In [0]:
df_spaark = spark.createDataFrame(df)
df_spaark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("indianhousing.silver.indian_housing_predictions")

display(df_spaark)